# RTX 4090 Laptop — carte CUDA courte

Mesures CUDA de cette machine : **Ada 8.9**, **76 SM**, **9 728 cœurs CUDA**, **304 Tensor Cores**, **76 RT Cores**, **16 376 Mio ≈ 15,99 Gio de VRAM**, bus **256 bits**, bande passante théorique **≈ 576 Go/s**.

| Par SM | Valeur |
|---|---:|
| Threads / warps résidents | 1 536 / 48 |
| Blocs résidents | 24 max. |
| Threads par bloc | 1 024 max. |
| Shared memory | 100 Kio par SM ; 48 Kio par bloc, 99 Kio avec opt-in |
| Registres | 65 536 registres de 32 bits = 256 Kio ; 255 max. par thread |

Le nombre réel de blocs résidents est le minimum imposé par les blocs (24), threads, warps, registres et shared memory.

## Unités de calcul et chemin mémoire

- **Cœurs CUDA** : calcul scalaire général (arithmétique des kernels).
- **Tensor Cores** : produits-accumulations de matrices, surtout IA et algèbre linéaire en précision réduite.
- **RT Cores** : parcours de structures BVH et intersections de rayons pour le ray tracing.

```text
VRAM GDDR6 -- bus 256 bits --> contrôleurs mémoire <--> L2 64 Mio
                                                       |
                                         76 SM : L1 / shared / registres
                                                       |
                                             CUDA / Tensor / RT
```

Le **bus mémoire** est la largeur de la liaison GDDR6 : 256 bits = 32 octets transférables en parallèle par battement. La **bande passante** est le débit maximal sur cette liaison. Les SM passent normalement par L1 puis L2 : la VRAM n'est pas reliée directement aux unités de calcul.

Le **L2**, partagé par tous les SM, évite des accès GDDR6 répétés et réduit trafic et latence. Le **L1** est consulté automatiquement pour les accès éligibles ; on ne le remplit généralement pas à la main. La **shared memory**, elle, est explicitement organisée par le kernel lorsqu'on veut maîtriser la réutilisation et la coopération entre threads d'un bloc.

In [ ]:
# Go (décimal) contre Gio (binaire)
octets_par_go = 10**9
octets_par_gio = 2**30
print(f"1 Go  = {octets_par_go:,} octets = {8 * octets_par_go:,} bits")
print(f"1 Gio = {octets_par_gio:,} octets = {8 * octets_par_gio:,} bits")

In [ ]:
# Un float32 occupe 4 octets
for nom, octets in {"shared/SM (100 Kio)": 102_400,
                    "shared/bloc standard (48 Kio)": 49_152,
                    "shared/bloc opt-in (99 Kio)": 101_376}.items():
    print(f"{nom}: {octets // 4:,} float32")